# Retrieval configuration evaluation

Compare **Vector**, **Hybrid (Vector + BM25)**, and **Hybrid + Reranking** to choose a shared retrieval configuration for the agent experiments.

We use 48 labelled questions and evaluate one retrieval call per question. This notebook does not generate answers or run agents. The same questions are used to compare and select configurations, so these are exploratory results, not held-out performance.


In [12]:
import hashlib
import json
import math
import sys
from dataclasses import asdict, replace
from pathlib import Path
from datetime import datetime, timezone

# Find the project root from results/rag/ or any other project subfolder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "SupportOpsBot/runtime/shared/config.py").is_file()), None)
if repo is None:
    raise RuntimeError("Start Jupyter within the project directory")
sys.path.insert(0, str(repo))

import pandas as pd
from IPython.display import display
from SupportOpsBot.runtime.shared.config import ROOT, load_config
from SupportOpsBot.runtime.shared.policy.rag.embeddings import read_embeddings
from SupportOpsBot.runtime.shared.policy.rag.retriever import APPROACHES, RagSettings, shared_retriever

base_settings = RagSettings.from_config(load_config()["policy"]["rag"])
manifest, documents = read_embeddings()
question_path = ROOT / "data/cases/rag/questions.json"
cases = json.loads(question_path.read_text())
ids = {doc["_id"] for doc in documents}
assert cases and len({c["id"] for c in cases}) == len(cases)
for case in cases:
    gold = case["expected_chunk_ids"]
    assert gold and len(gold) == len(set(gold)) and set(gold) <= ids, case["id"]
    assert set(case.get("expected_reference_chunk_ids", [])) <= ids, case["id"]
print(f"{len(cases)} questions; {len(ids)} chunks")


48 questions; 52 chunks


## Configurations

| Method | Candidates retrieved (N′) | Final ranked chunks (k) |
|---|---|---|
| Vector | 20 (fixed) | 4, 5, 6, 7, 8 |
| Hybrid (Vector + BM25) | 20 (fixed) | 4, 5, 6, 7, 8 |
| Hybrid + Reranking | 8, 12, 20 | 4, 5, 6, 7, 8 |

- **N′**: number of candidates requested from Pinecone for each search.
- **k**: number of ranked chunks selected and returned to the agent, before adding linked references.
- **Vector**: retrieve N′ candidates and select the top k.
- **Hybrid (Vector + BM25)**: retrieve N′ candidates from each search, combine their rankings using Reciprocal Rank Fusion (RRF), keep the top N′, and select the top k.
- **Hybrid + Reranking**: combine the rankings using RRF, keep the top N′, then use **BAAI/bge-reranker-v2-m3** to score each candidate against the question and reorder them. Select the top k.

N′ is a requested maximum; Pinecone may return fewer if fewer matches are available.

Changing N′ for reranking changes both search depth and the reranking pool size.

This gives **25 configurations**. Linked references are added after selecting k, so the agent may receive more than k chunks. Other settings stay fixed; the runtime config is unchanged.

### RRF formula

For a chunk $d$, combine its positions in the Vector and BM25 rankings:

$$
\operatorname{RRF}(d) = \sum_{s \in \{\text{Vector},\text{BM25}\}:\ d \in s}
\frac{1}{c + \operatorname{rank}_s(d)}
$$

Ranks start at 1. A chunk contributes only for searches where it appears. Higher scores rank first. The constant $c$ is `base_settings.rrf_k` (60 in the shared default config), not the final chunk count k. RRF runs in the shared retriever's `reciprocal_rank_fusion` function; this notebook calls that retriever.


In [13]:
K_VALUES = (4, 5, 6, 7, 8)
RERANK_POOLS = (8, 12, 20)
FIXED_SEARCH_DEPTH = 20
configurations = []
for method in APPROACHES:
    for pool in (RERANK_POOLS if method == "hybrid_rerank" else (FIXED_SEARCH_DEPTH,)):
        for k in K_VALUES:
            settings = replace(base_settings, **APPROACHES[method], k=k,
                               n_prime=pool, expand_references=True)
            configurations.append({"config_id": f"{method}_k{k}_depth{pool}",
                                   "method": method, "settings": settings})
assert len(configurations) == 25
assert max(RERANK_POOLS) <= len(ids)
display(pd.DataFrame([{"config_id": c["config_id"], "method": c["method"],
                       "k": c["settings"].k, "candidate_depth": c["settings"].n_prime,
                       "rerank_pool": c["settings"].n_prime if c["settings"].rerank else None}
                      for c in configurations]))


,config_id,method,k,candidate_depth,rerank_pool
0,vector_k4_depth20,vector,4,20,NaN
1,vector_k5_depth20,vector,5,20,NaN
2,vector_k6_depth20,vector,6,20,NaN
3,vector_k7_depth20,vector,7,20,NaN
4,vector_k8_depth20,vector,8,20,NaN
5,hybrid_k4_depth20,hybrid,4,20,NaN
6,hybrid_k5_depth20,hybrid,5,20,NaN
7,hybrid_k6_depth20,hybrid,6,20,NaN
8,hybrid_k7_depth20,hybrid,7,20,NaN
9,hybrid_k8_depth20,hybrid,8,20,NaN


## Evaluation metrics

We use three metrics (higher is better):

| Metric | What it measures |
|---|---|
| **Recall@k** | Out of all expected chunks defined by our gold dataset for a question, what fraction were successfully retrieved in the top k? |
| **nDCG@k** | Overall quality of the ranking order in the top k, giving lower scores when expected chunks from the gold dataset appear further down the list. |
| **Complete evidence coverage (primary)** | Fraction of questions where the final retrieval response contains every expected chunk in the gold dataset, including labelled references. |

Recall@k and nDCG@k use the main gold chunks in `expected_chunk_ids`, before reference expansion. Both are calculated per question and averaged across questions. nDCG uses binary labels: each main gold chunk has equal relevance.

Complete evidence coverage checks both `expected_chunk_ids` and `expected_reference_chunk_ids` after expansion. We also report **returned clauses** to show final context size.

These scores measure retrieval of labelled evidence; they do not establish that unlabelled chunks are irrelevant. We do not use Precision@k or MRR.


In [14]:
def score_response(case, hits, k):
    ranked = list(dict.fromkeys(h["id"] for h in hits if h["via"] == "search"))[:k]
    returned = {h["id"] for h in hits}
    main_gold = set(case["expected_chunk_ids"])
    gold = main_gold | set(case.get("expected_reference_chunk_ids", []))
    relevant = [int(chunk_id in main_gold) for chunk_id in ranked]
    dcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(relevant))
    ideal_dcg = sum(1 / math.log2(rank + 2) for rank in range(min(k, len(main_gold))))
    return {"retrieved_chunk_ids": ranked,
            "reference_chunk_ids": [h["id"] for h in hits if h["via"] != "search"],
            "recall_at_k": sum(relevant) / len(main_gold),
            "ndcg_at_k": dcg / ideal_dcg,
            "complete_evidence": gold <= returned,
            "missing_evidence_ids": sorted(gold - returned),
            "returned_clauses": len(returned)}


## Run the comparison

Requires the existing Pinecone snapshot, API credentials, and local embedding/reranking models. Runs 25 configurations across all questions without changing the index. Run this cell to generate results.


In [15]:
rows = []
for number, config in enumerate(configurations, 1):
    print(f"[{number}/{len(configurations)}] {config['config_id']}", flush=True)
    retriever = shared_retriever(config["settings"])
    for case in cases:
        hits = retriever.search(case["question"])
        rows.append({"config_id": config["config_id"], "method": config["method"],
                     "k": config["settings"].k, "candidate_depth": config["settings"].n_prime,
                     "case_id": case["id"], "question": case["question"],
                     "expected_chunk_ids": case["expected_chunk_ids"],
                     "expected_reference_chunk_ids": case.get("expected_reference_chunk_ids", []),
                     **score_response(case, hits, config["settings"].k)})
results = pd.DataFrame(rows)
assert len(results) == len(configurations) * len(cases)


[1/25] vector_k4_depth20
[2/25] vector_k5_depth20
[3/25] vector_k6_depth20
[4/25] vector_k7_depth20
[5/25] vector_k8_depth20
[6/25] hybrid_k4_depth20
[7/25] hybrid_k5_depth20
[8/25] hybrid_k6_depth20
[9/25] hybrid_k7_depth20
[10/25] hybrid_k8_depth20
[11/25] hybrid_rerank_k4_depth8
[12/25] hybrid_rerank_k5_depth8
[13/25] hybrid_rerank_k6_depth8
[14/25] hybrid_rerank_k7_depth8
[15/25] hybrid_rerank_k8_depth8
[16/25] hybrid_rerank_k4_depth12
[17/25] hybrid_rerank_k5_depth12
[18/25] hybrid_rerank_k6_depth12
[19/25] hybrid_rerank_k7_depth12
[20/25] hybrid_rerank_k8_depth12
[21/25] hybrid_rerank_k4_depth20
[22/25] hybrid_rerank_k5_depth20
[23/25] hybrid_rerank_k6_depth20
[24/25] hybrid_rerank_k7_depth20
[25/25] hybrid_rerank_k8_depth20


## Compare and select

Select the configuration with the highest **complete evidence coverage**, then **Recall@k**, then **nDCG@k**. Break remaining ties by fewer returned clauses, smaller k, then smaller N′.

This prioritises complete evidence, then main-chunk coverage and ranking quality. It selects the best configuration on this question set; agent performance is evaluated separately.

The full table compares all 25 configurations. The final summary keeps the best configuration for each method (three rows), using the same selection rule.


In [16]:
keys = ["config_id", "method", "k", "candidate_depth"]
metrics = ["complete_evidence", "recall_at_k", "ndcg_at_k", "returned_clauses"]
summary = results.groupby(keys, sort=False)[metrics].mean().reset_index()
summary["questions"] = len(cases)
summary = summary.sort_values(
    ["complete_evidence", "recall_at_k", "ndcg_at_k",
     "returned_clauses", "k", "candidate_depth", "config_id"],
    ascending=[False, False, False, True, True, True, True],
).reset_index(drop=True)
selected_id = summary.iloc[0]["config_id"]
selected = next(c for c in configurations if c["config_id"] == selected_id)
METHOD_NAMES = {
    "vector": "Vector",
    "hybrid": "Hybrid (Vector + BM25)",
    "hybrid_rerank": "Hybrid + Reranking",
}
COLUMN_NAMES = {
    "method": "Method", "candidate_depth": "N′",
    "complete_evidence": "Complete evidence coverage",
    "recall_at_k": "Recall@k", "ndcg_at_k": "nDCG@k",
    "returned_clauses": "Returned clauses",
}
print("All configurations:")
display(summary.replace({"method": METHOD_NAMES}).rename(columns=COLUMN_NAMES).round(4))

# summary is already sorted by the selection rule; first per method is its best.
final_retrieval_summary = (summary.drop_duplicates("method")
                           .set_index("method").loc[list(METHOD_NAMES)].reset_index())
final_retrieval_summary = final_retrieval_summary[[
    "method", "k", "candidate_depth", "recall_at_k", "ndcg_at_k", "complete_evidence"
]].replace({"method": METHOD_NAMES}).rename(columns=COLUMN_NAMES)
assert len(final_retrieval_summary) == 3
print("Best configuration per method:")
display(final_retrieval_summary.round(4))
print("Selected by the stated rule:", selected_id)
print("Proposed shared RAG settings (not applied):")
print(json.dumps(asdict(selected["settings"]), indent=2))


All configurations:


,config_id,Method,k,N′,Complete evidence coverage,Recall@k,nDCG@k,Returned clauses,questions
0,hybrid_rerank_k8_depth20,Hybrid + Reranking,8,20,0.9792,0.9757,0.9199,14.4375,48
1,hybrid_rerank_k6_depth20,Hybrid + Reranking,6,20,0.9792,0.9687,0.9169,10.8750,48
2,hybrid_rerank_k7_depth20,Hybrid + Reranking,7,20,0.9792,0.9687,0.9169,12.5000,48
3,vector_k8_depth20,Vector,8,20,0.9583,0.9722,0.8473,13.5833,48
4,hybrid_rerank_k8_depth12,Hybrid + Reranking,8,12,0.9583,0.9618,0.9103,14.5625,48
5,hybrid_rerank_k6_depth12,Hybrid + Reranking,6,12,0.9583,0.9549,0.9072,11.5833,48
6,hybrid_rerank_k7_depth12,Hybrid + Reranking,7,12,0.9583,0.9549,0.9072,13.1667,48
7,vector_k7_depth20,Vector,7,20,0.9375,0.9549,0.8402,12.2083,48
8,hybrid_rerank_k5_depth20,Hybrid + Reranking,5,20,0.9375,0.9549,0.9099,9.4583,48
9,hybrid_rerank_k8_depth8,Hybrid + Reranking,8,8,0.9375,0.9514,0.9034,14.6458,48


Best configuration per method:


,Method,k,N′,Recall@k,nDCG@k,Complete evidence coverage
0,Vector,8,20,0.9722,0.8473,0.9583
1,Hybrid (Vector + BM25),8,20,0.9097,0.8024,0.9167
2,Hybrid + Reranking,8,20,0.9757,0.9199,0.9792


Selected by the stated rule: hybrid_rerank_k8_depth20
Proposed shared RAG settings (not applied):
{
  "dense": true,
  "bm25": true,
  "rrf_k": 60,
  "n_prime": 20,
  "rerank": true,
  "contextual": true,
  "k": 8,
  "expand_references": true,
  "embedding_model": "nomic-ai/nomic-embed-text-v1.5",
  "rerank_model": "BAAI/bge-reranker-v2-m3"
}


In [17]:
# Inspect one original question across all configurations, including missing evidence.
CASE_ID = "q41"
print(next(c["question"] for c in cases if c["id"] == CASE_ID))
with pd.option_context("display.max_colwidth", None):
    display(results.loc[results.case_id == CASE_ID, [
        "config_id", "expected_chunk_ids", "retrieved_chunk_ids", "reference_chunk_ids",
        "missing_evidence_ids", "complete_evidence", "recall_at_k",
        "ndcg_at_k", "returned_clauses"
    ]].reset_index(drop=True))


My $20 refund has been accepted and the item inspection is complete. Can it be processed automatically?


,config_id,expected_chunk_ids,retrieved_chunk_ids,reference_chunk_ids,missing_evidence_ids,complete_evidence,recall_at_k,ndcg_at_k,returned_clauses
0,vector_k4_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.5]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.2, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3]",[],True,1.0,0.63093,12
1,vector_k5_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.5, RET-ReturnParcelDisputes-7.2]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3]",[],True,1.0,0.63093,12
2,vector_k6_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.5, RET-ReturnParcelDisputes-7.2, ODR-OrderChangeCancel-4.2]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3]",[],True,1.0,0.63093,13
3,vector_k7_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.5, RET-ReturnParcelDisputes-7.2, ODR-OrderChangeCancel-4.2, RET-ReturnProcedure-4.1]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3, RET-ReturnEligibility-2.1, RET-ReturnEligibility-2.2, RET-ReturnEligibility-2.3, RET-ReturnEligibility-2.4, RET-SupportingEvidence-3.1, RET-SupportingEvidence-3.2]",[],True,1.0,0.63093,20
4,vector_k8_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.5, RET-ReturnParcelDisputes-7.2, ODR-OrderChangeCancel-4.2, RET-ReturnProcedure-4.1, ODR-OrderChangeCancel-4.1]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3, RET-ReturnEligibility-2.1, RET-ReturnEligibility-2.2, RET-ReturnEligibility-2.3, RET-ReturnEligibility-2.4, RET-SupportingEvidence-3.1, RET-SupportingEvidence-3.2]",[],True,1.0,0.63093,21
5,hybrid_k4_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.1]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.2, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3, RET-ReturnEligibility-2.1, RET-ReturnEligibility-2.2, RET-ReturnEligibility-2.3, RET-ReturnEligibility-2.4, RET-SupportingEvidence-3.1, RET-SupportingEvidence-3.2]",[],True,1.0,0.63093,18
6,hybrid_k5_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.1, ODR-OrderChangeCancel-4.2]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.2, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3, RET-ReturnEligibility-2.1, RET-ReturnEligibility-2.2, RET-ReturnEligibility-2.3, RET-ReturnEligibility-2.4, RET-SupportingEvidence-3.1, RET-SupportingEvidence-3.2]",[],True,1.0,0.63093,19
7,hybrid_k6_depth20,[REF-Refunds-6.6],"[REF-Refunds-6.1, REF-Refunds-6.6, RET-ReturnEligibility-2.6, RET-ReturnProcedure-4.1, ODR-OrderChangeCancel-4.2, RET-ReturnParcelDisputes-7.2]","[RET-ReturnEligibility-2.5, RET-ReturnParcelDisputes-7.1, RET-ReturnParcelDisputes-7.3, RET-ReturnParcelDisputes-7.4, RET-ReturnParcelDisputes-7.5, REF-Refunds-6.2, REF-Refunds-6.3, RET-ReturnEligibility-2.1, RET-ReturnEligibility-2.2, RET-ReturnEligibility-2.3, RET-ReturnEligi

## Save results

Run the next cell after the comparison to save detailed results, all 25 configuration scores, and the three-row `final_retrieval_summary` (CSV and JSON) in `results/rag/`.

These are selection-set retrieval scores. Apply the chosen runtime settings separately before agent experiments.


In [18]:
SAVE_RESULTS = True  # Save when this cell is explicitly run after the comparison.
if SAVE_RESULTS:
    output = ROOT / "results/rag"
    output.mkdir(parents=True, exist_ok=True)
    payload = {"created_utc": datetime.now(timezone.utc).isoformat(),
               "corpus_hash": manifest["corpus_hash"],
               "questions_sha256": hashlib.sha256(question_path.read_bytes()).hexdigest(),
               "question_count": len(cases), "selection_set_only": True,
               "configurations": [{"config_id": c["config_id"], "method": c["method"],
                                    "settings": asdict(c["settings"])} for c in configurations],
               "selected_config_id": selected_id, "rows": rows}
    (output / "configuration_results.json").write_text(json.dumps(payload, indent=2) + "\n")
    summary.to_json(output / "configuration_summary.json", orient="records", indent=2)
    final_retrieval_summary.to_csv(output / "final_retrieval_summary.csv", index=False)
    final_retrieval_summary.to_json(output / "final_retrieval_summary.json", orient="records", indent=2)
